# 🗺️ Arricchimento Excel con dati Mapillary — Map Features API v4
# ✅ Versione corretta con Checkpoint / Resume

---

## Cosa fa questo notebook

Prende in input il file Excel della object detection (struttura: 4 righe orientamento + 1 riga TOTALE
per ogni PANO ID) e aggiunge colonne con i conteggi degli oggetti Mapillary entro **25 metri**
da ogni punto geografico. I dati vengono scritti **solo nelle righe TOTALE**.

### Sistema checkpoint
- Ogni **100 PANO ID** i risultati vengono salvati in un file JSON su Drive
- Se il runtime crasha, alla ripresa il notebook legge il checkpoint e riparte da dove si era fermato
- L'Excel di output viene aggiornato su Drive ad ogni checkpoint

### In caso di crash
1. Riconnetti il runtime
2. Riesegui **tutte le celle nell'ordine**
3. La Cella 5 (query) troverà il checkpoint e ripartirà automaticamente
4. **Non cancellare** `mapillary_checkpoint.json` dalla cartella di output

---

## CELLA 1 — Installazione pacchetti

In [ ]:
!pip install -q requests openpyxl pandas mercantile vt2geojson mapbox-vector-tile tqdm

import requests, math, time, json, os
import pandas as pd
import mercantile
from vt2geojson.tools import vt_bytes_to_geojson
from tqdm.notebook import tqdm
print('✅ Pacchetti installati correttamente')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.4/323.4 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 72.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.6 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 6.33.6 which is incompatible.
✅ Pacchetti installati correttamente


## CELLA 2 — Montaggio Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print('✅ Google Drive montato')

Mounted at /content/drive
✅ Google Drive montato


## CELLA 3 — ✏️ Configurazione

**Modifica solo questa cella** con i tuoi percorsi, token e classi.

In [ ]:
# =============================================================
# ✏️  TOKEN MAPILLARY
# =============================================================
MAPILLARY_TOKEN = '...'

# =============================================================
# ✏️  PERCORSI FILE
# =============================================================

# File Excel di input (quello della object detection)
INPUT_EXCEL_PATH = '...'

# Nome del foglio di input
INPUT_SHEET_NAME = '...'

# File Excel di output (il file di input arricchito con le colonne Mapillary)
OUTPUT_EXCEL_PATH = '...'

# Nome del foglio di output
OUTPUT_SHEET_NAME = '...'

# File JSON per i checkpoint (salvato nella stessa cartella dell'output)
CHECKPOINT_PATH = '...'

# =============================================================
# ✏️  NOMI COLONNE NEL TUO EXCEL (non modificare se usi il file standard)
# =============================================================
COL_LAT         = 'Latitudine'
COL_LNG         = 'Longitudine'
COL_PANO_ID     = 'PANO_ID'
COL_ORIENTATION = 'Orientamento'
SUMMARY_LABEL   = 'TOTALE'   # valore nella colonna Orientamento per le righe riassuntive

# =============================================================
# ✏️  PARAMETRI RICERCA
# =============================================================
SEARCH_RADIUS_M    = 25    # raggio in metri
API_DELAY          = 0.2   # secondi tra una richiesta e la successiva
CHECKPOINT_EVERY_N = 100   # salva checkpoint ogni N PANO ID

# =============================================================
# ✏️  CLASSI MAPILLARY DA CERCARE
#     Chiave  = nome colonna che apparirà nell'Excel
#     Valore  = valore API Mapillary
# =============================================================
MAPILLARY_CLASSES = {
    'mly_Bench':        'object--bench',
    'mly_Phone_Booth':  'object--phone-booth',
    'mly_Sign_Store':   'object--sign--store',
    'mly_Pole':         'object--support--pole',
    'mly_Utility_Pole': 'object--support--utility-pole',
    'mly_Trash_Can':    'object--trash-can',
}

# =============================================================

# Crea cartella output se non esiste
os.makedirs(os.path.dirname(OUTPUT_EXCEL_PATH), exist_ok=True)

print('✅ Configurazione completata')
print(f'   Input:              {INPUT_EXCEL_PATH}')
print(f'   Output:             {OUTPUT_EXCEL_PATH}')
print(f'   Checkpoint:         {CHECKPOINT_PATH}')
print(f'   Raggio ricerca:     {SEARCH_RADIUS_M} m')
print(f'   Checkpoint ogni:    {CHECKPOINT_EVERY_N} PANO ID')
print(f'   Label riga TOTALE:  "{SUMMARY_LABEL}"')
print(f'\n🗂️  Classi Mapillary ({len(MAPILLARY_CLASSES)}):')
for col, val in MAPILLARY_CLASSES.items():
    print(f'   {col:<25} → {val}')

✅ Configurazione completata
   Input:              /content/drive/MyDrive/Polimi/SVBEF/Progetto Finale/c_Definitivo/b_Outputs_Object Detection/Outputs_Excel Object Detection - Inputs_Excel Script Mapillary/Results_ObjectDetection Definitivo.xlsx
   Output:             /content/drive/MyDrive/Polimi/SVBEF/Progetto Finale/c_Definitivo/c_Outputs_Excel Script Mapillary - Inputs_Excel Script Excel con FI/Results_ObjectDetection+Mapillary Definitivo.xlsx
   Checkpoint:         /content/drive/MyDrive/Polimi/SVBEF/Progetto Finale/c_Definitivo/c_Outputs_Excel Script Mapillary - Inputs_Excel Script Excel con FI/mapillary_checkpoint.json
   Raggio ricerca:     25 m
   Checkpoint ogni:    100 PANO ID
   Label riga TOTALE:  "TOTALE"

🗂️  Classi Mapillary (7):
   mly_Banner                → object--banner
   mly_Bench                 → object--bench
   mly_Phone_Booth           → object--phone-booth
   mly_Sign_Store            → object--sign--store
   mly_Pole                  → object--support--pol

## CELLA 4 — Lettura Excel e funzioni di supporto

In [ ]:
# ---------------------------------------------------------------
# LETTURA EXCEL
# ---------------------------------------------------------------
df_all = pd.read_excel(INPUT_EXCEL_PATH, sheet_name=INPUT_SHEET_NAME, dtype=str)

print(f'✅ File Excel letto')
print(f'   Righe totali: {len(df_all)}')
print(f'   Colonne:      {list(df_all.columns)}')

# Verifica colonne necessarie
for col in [COL_LAT, COL_LNG, COL_PANO_ID, COL_ORIENTATION]:
    assert col in df_all.columns, \
        f'❌ Colonna "{col}" non trovata. Disponibili: {list(df_all.columns)}'

# Isola le righe TOTALE — queste sono i punti geografici da interrogare
# Conserviamo l'indice originale del DataFrame completo: serve per scrivere
# i risultati nelle righe giuste al momento dell'esportazione
mask_totale   = df_all[COL_ORIENTATION].str.strip() == SUMMARY_LABEL
df_totale     = df_all[mask_totale].copy()
df_totale[COL_LAT] = pd.to_numeric(df_totale[COL_LAT], errors='coerce')
df_totale[COL_LNG] = pd.to_numeric(df_totale[COL_LNG], errors='coerce')
df_totale = df_totale.dropna(subset=[COL_LAT, COL_LNG])

# Lista ordinata di (indice_originale, pano_id, lat, lng)
query_list = [
    (orig_idx, row[COL_PANO_ID], float(row[COL_LAT]), float(row[COL_LNG]))
    for orig_idx, row in df_totale.iterrows()
]

print(f'\n✅ Punti geografici (righe TOTALE) da interrogare: {len(query_list)}')
print(f'   Esempi valori nella colonna Orientamento: {df_all[COL_ORIENTATION].unique()[:8]}')
print(f'\n📋 Primi 5 punti:')
for orig_idx, pano, lat, lng in query_list[:5]:
    print(f'   idx={orig_idx}  PANO={pano}  lat={lat:.5f}  lng={lng:.5f}')

# ---------------------------------------------------------------
# FUNZIONI DI SUPPORTO
# ---------------------------------------------------------------
def meters_to_degrees(meters, latitude):
    lat_deg = meters / 111320.0
    lng_deg = meters / (111320.0 * math.cos(math.radians(latitude)))
    return lat_deg, lng_deg

def haversine_distance(lat1, lng1, lat2, lng2):
    R    = 6371000
    phi1 = math.radians(lat1); phi2 = math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlam = math.radians(lng2 - lng1)
    a    = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlam/2)**2
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))

def query_mapillary_features(lat, lng, radius_m, object_values, token, zoom=14):
    lat_d, lng_d = meters_to_degrees(radius_m, lat)
    west, east   = lng - lng_d, lng + lng_d
    south, north = lat - lat_d, lat + lat_d
    tiles        = list(mercantile.tiles(west, south, east, north, zoom))
    counts       = {v: 0 for v in object_values}
    seen_ids     = set()

    for tile in tiles:
        url = (f'https://tiles.mapillary.com/maps/vtp/mly_map_feature_point/2/'
               f'{tile.z}/{tile.x}/{tile.y}?access_token={token}')
        try:
            resp = requests.get(url, timeout=15)
            if resp.status_code != 200:
                continue
            geojson = vt_bytes_to_geojson(resp.content, tile.x, tile.y, tile.z)
            for feat in geojson.get('features', []):
                props   = feat.get('properties', {})
                fid     = props.get('id') or feat.get('id')
                obj_val = props.get('value', '')
                if obj_val not in object_values: continue
                if fid and fid in seen_ids:      continue
                if fid: seen_ids.add(fid)
                coords = feat.get('geometry', {}).get('coordinates', [])
                if len(coords) < 2: continue
                if haversine_distance(lat, lng, coords[1], coords[0]) <= radius_m:
                    counts[obj_val] += 1
        except Exception:
            pass
    return counts

print('\n✅ Funzioni di supporto definite')

✅ File Excel letto
   Righe totali: 23550
   Colonne:      ['Latitudine', 'Longitudine', 'PANO_ID', 'Orientamento', 'awning', 'canopy', 'fence', 'billboard', 'bus stop shelter', 'sculpture', 'monument', 'fountain', 'umbrella']

✅ Punti geografici (righe TOTALE) da interrogare: 4710
   Esempi valori nella colonna Orientamento: ['0°' '90°' '180°' '270°' 'TOTALE']

📋 Primi 5 punti:
   idx=4  PANO=--tlUSu9Lrgn80E_kwY5-g  lat=45.45118  lng=9.20638
   idx=9  PANO=-0CpaWIBx87rZ1dDb2bVxQ  lat=45.48002  lng=9.17197
   idx=14  PANO=-0WwME3Z-gqiRsNJNJ1uUw  lat=45.47933  lng=9.19460
   idx=19  PANO=-0bQMlJHsAZy_XedjVdmjA  lat=45.45520  lng=9.16940
   idx=24  PANO=-19fTsUM-o2rNyD3LGY3ww  lat=45.45299  lng=9.17728

✅ Funzioni di supporto definite


## CELLA 5 — Query Mapillary con Checkpoint / Resume

Questa cella:
- Legge il checkpoint se esiste e riparte da dove si era fermata
- Interroga Mapillary per ogni punto geografico (righe TOTALE)
- Salva checkpoint ogni 100 PANO ID
- Aggiorna l'Excel su Drive ad ogni checkpoint

In [ ]:
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

object_values_list = list(MAPILLARY_CLASSES.values())

# ---------------------------------------------------------------
# CARICAMENTO CHECKPOINT
# results_map: {indice_originale_excel (str) → {api_value: conteggio}}
# ---------------------------------------------------------------
results_map      = {}   # chiave = str(orig_idx) per serializzazione JSON
completed_panos  = set()

if os.path.exists(CHECKPOINT_PATH):
    print(f'📂 Checkpoint trovato: {CHECKPOINT_PATH}')
    try:
        with open(CHECKPOINT_PATH, 'r') as f:
            ckpt = json.load(f)
        results_map     = ckpt.get('results_map', {})
        completed_panos = set(ckpt.get('completed_panos', []))
        print(f'   ✅ Resume da checkpoint:')
        print(f'      PANO ID già completati: {len(completed_panos)}')
        print(f'      PANO ID rimanenti:      {len(query_list) - len(completed_panos)}')
    except Exception as e:
        print(f'   ⚠️  Errore lettura checkpoint ({e}) — si riparte da zero')
        results_map     = {}
        completed_panos = set()
else:
    print('🆕 Nessun checkpoint — avvio da zero')

# ---------------------------------------------------------------
# FUNZIONE SALVATAGGIO CHECKPOINT + EXCEL INTERMEDIO
# ---------------------------------------------------------------
def save_checkpoint_and_excel():
    # 1. Salva JSON (scrittura atomica)
    ckpt = {'results_map': results_map, 'completed_panos': list(completed_panos)}
    tmp  = CHECKPOINT_PATH + '.tmp'
    with open(tmp, 'w') as f:
        json.dump(ckpt, f)
    os.replace(tmp, CHECKPOINT_PATH)

    # 2. Aggiorna Excel
    try:
        write_output_excel()
    except Exception as e:
        print(f'   ⚠️  Errore aggiornamento Excel: {e}')

# ---------------------------------------------------------------
# FUNZIONE SCRITTURA EXCEL DI OUTPUT
# Costruisce il DataFrame completo e lo salva con formattazione
# ---------------------------------------------------------------
def write_output_excel():
    # Rileggi il file originale
    df_out = pd.read_excel(INPUT_EXCEL_PATH, sheet_name=INPUT_SHEET_NAME, dtype=str)

    # Aggiungi colonne Mapillary vuote
    for col_name in MAPILLARY_CLASSES.keys():
        df_out[col_name] = ''

    # Popola le righe TOTALE con i risultati
    for orig_idx, pano, lat, lng in query_list:
        key = str(orig_idx)
        if key not in results_map:
            continue
        counts = results_map[key]
        for col_name, api_val in MAPILLARY_CLASSES.items():
            df_out.at[orig_idx, col_name] = counts.get(api_val, 0)

    # Salva su Excel
    df_out.to_excel(OUTPUT_EXCEL_PATH, index=False, sheet_name=OUTPUT_SHEET_NAME)

    # Formattazione openpyxl
    wb = load_workbook(OUTPUT_EXCEL_PATH)
    ws = wb[OUTPUT_SHEET_NAME]

    hdr_blue   = PatternFill(start_color='1F4E79', end_color='1F4E79', fill_type='solid')
    hdr_green  = PatternFill(start_color='375623', end_color='375623', fill_type='solid')
    hdr_font   = Font(bold=True, color='FFFFFF', size=10)
    tot_fill   = PatternFill(start_color='FFF2CC', end_color='FFF2CC', fill_type='solid')
    tot_mly    = PatternFill(start_color='E2EFDA', end_color='E2EFDA', fill_type='solid')
    row_fill   = PatternFill(start_color='FFFFFF', end_color='FFFFFF', fill_type='solid')
    tot_font   = Font(bold=True, size=9)
    row_font   = Font(size=9)
    thin_brd   = Border(bottom=Side(style='thin',   color='BFBFBF'))
    thick_brd  = Border(bottom=Side(style='medium', color='1F4E79'))

    n_mly      = len(MAPILLARY_CLASSES)
    total_cols = len(df_out.columns)
    mly_start  = total_cols - n_mly + 1  # 1-indexed
    ori_col    = list(df_out.columns).index(COL_ORIENTATION) + 1  # 1-indexed

    # Intestazione
    for c_idx, cell in enumerate(ws[1], start=1):
        cell.fill      = hdr_green if c_idx >= mly_start else hdr_blue
        cell.font      = hdr_font
        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
    ws.row_dimensions[1].height = 35

    # Righe dati
    for r_idx in range(2, ws.max_row + 1):
        is_tot = str(ws.cell(row=r_idx, column=ori_col).value).strip() == SUMMARY_LABEL
        for c_idx in range(1, total_cols + 1):
            cell        = ws.cell(row=r_idx, column=c_idx)
            is_mly      = c_idx >= mly_start
            cell.font   = tot_font if is_tot else row_font
            cell.border = thick_brd if is_tot else thin_brd
            cell.alignment = Alignment(horizontal='center', vertical='center')
            if is_tot:
                cell.fill = tot_mly if is_mly else tot_fill
            else:
                cell.fill = row_fill

    # Larghezze colonne
    preset = {'Latitudine':14,'Longitudine':14,'PANO_ID':28,'Orientamento':22}
    for c_idx, col_name in enumerate(df_out.columns, start=1):
        ws.column_dimensions[get_column_letter(c_idx)].width = preset.get(
            col_name, max(14, len(col_name) + 2))

    wb.save(OUTPUT_EXCEL_PATH)

# ---------------------------------------------------------------
# LOOP PRINCIPALE
# ---------------------------------------------------------------
todo        = [(i, p, la, ln) for i, p, la, ln in query_list if p not in completed_panos]
pano_count  = 0
errors      = []

print(f'\n🚀 Avvio query Mapillary')
print(f'   Punti totali:      {len(query_list)}')
print(f'   Già completati:    {len(completed_panos)}')
print(f'   Da elaborare:      {len(todo)}')
print(f'   Checkpoint ogni:   {CHECKPOINT_EVERY_N} PANO ID')
print('-' * 70)

for orig_idx, pano, lat, lng in tqdm(todo, desc='Query Mapillary'):
    try:
        counts = query_mapillary_features(
            lat=lat, lng=lng,
            radius_m=SEARCH_RADIUS_M,
            object_values=object_values_list,
            token=MAPILLARY_TOKEN
        )
        # Salva usando str(orig_idx) come chiave (JSON richiede chiavi stringa)
        results_map[str(orig_idx)] = counts
        completed_panos.add(pano)

        found   = {k: v for k, v in counts.items() if v > 0}
        summary = ', '.join([f"{k.split('--')[-1]}:{v}" for k, v in found.items()])
        tot     = sum(counts.values())
        tqdm.write(f'  {pano[:30]:<30} ({lat:.5f}, {lng:.5f})  →  {tot} obj: {summary if summary else "nessuno"}')

    except Exception as e:
        results_map[str(orig_idx)] = {v: 0 for v in object_values_list}
        completed_panos.add(pano)
        errors.append({'pano': pano, 'error': str(e)})
        tqdm.write(f'  ❌ Errore su {pano}: {e}')

    pano_count += 1
    time.sleep(API_DELAY)

    # Checkpoint ogni N PANO ID
    if pano_count % CHECKPOINT_EVERY_N == 0:
        tqdm.write(f'\n💾 Checkpoint: {len(completed_panos)}/{len(query_list)} — salvataggio su Drive...')
        save_checkpoint_and_excel()
        tqdm.write(f'   ✅ Salvato.\n')

# Salvataggio finale
print('\n💾 Salvataggio finale...')
save_checkpoint_and_excel()

print(f'\n✅ Query completate: {len(completed_panos)}/{len(query_list)} PANO ID')
if errors:
    print(f'⚠️  Errori ({len(errors)}):')
    for e in errors[:10]:
        print(f'   - {e["pano"]}: {e["error"]}')

📂 Checkpoint trovato: /content/drive/MyDrive/Polimi/SVBEF/Progetto Finale/c_Definitivo/c_Outputs_Excel Script Mapillary - Inputs_Excel Script Excel con FI/mapillary_checkpoint.json
   ✅ Resume da checkpoint:
      PANO ID già completati: 1300
      PANO ID rimanenti:      3410

🚀 Avvio query Mapillary
   Punti totali:      4710
   Già completati:    1300
   Da elaborare:      3410
   Checkpoint ogni:   100 PANO ID
----------------------------------------------------------------------


Query Mapillary:   0%|          | 0/3410 [00:00<?, ?it/s]

  FUZubMDJexjArSSbzzJ-dg         (45.46907, 9.16985)  →  4 obj: bench:1, trash-can:3
  FUtFA9iO7EpT9RnCXNy-zw         (45.46440, 9.16524)  →  8 obj: banner:2, store:3, pole:1, trash-can:2
  FVM53qlO1ouJxivSLJIyeg         (45.46811, 9.20592)  →  3 obj: store:3
  FWrqIXqkLaLkUkw9A2vnyw         (45.48254, 9.21002)  →  2 obj: trash-can:2
  FXbmmafb5KLD2zXXmSJooQ         (45.47997, 9.18998)  →  4 obj: trash-can:4
  FYCYdfe4yGM44kP6p_LL9Q         (45.45518, 9.20105)  →  0 obj: nessuno
  FZbdKthkBmmcFjiVE2XDGA         (45.47763, 9.16738)  →  3 obj: banner:2, trash-can:1
  F_0b-5G0z9_SLnJ14f8rLw         (45.45188, 9.17911)  →  5 obj: banner:1, store:1, trash-can:3
  Fb-4Mbh3vaCd6qENe68YPw         (45.47960, 9.19492)  →  19 obj: store:11, utility-pole:5, trash-can:3
  FbQOmDnTeotNyRE_3ut-ng         (45.48113, 9.18216)  →  7 obj: banner:3, pole:1, utility-pole:3
  Feaxjm_5D8DZPSVOCQfBfA         (45.46573, 9.17369)  →  5 obj: store:3, trash-can:2
  FfWS2S2YWMOsDypX1MCcog         (45.45776, 9.2046

## CELLA 6 — Riepilogo statistico finale

In [ ]:
# Leggi il file di output appena scritto per il riepilogo
df_out   = pd.read_excel(OUTPUT_EXCEL_PATH, sheet_name=OUTPUT_SHEET_NAME, dtype=str)
df_tot   = df_out[df_out[COL_ORIENTATION].str.strip() == SUMMARY_LABEL].copy()

for col in MAPILLARY_CLASSES.keys():
    if col in df_tot.columns:
        df_tot[col] = pd.to_numeric(df_tot[col], errors='coerce').fillna(0).astype(int)

print('=' * 72)
print(f'📈 RIEPILOGO MAPILLARY — oggetti rilevati entro {SEARCH_RADIUS_M}m per punto')
print('=' * 72)
print(f'   Punti geografici interrogati: {len(df_tot)}')
print(f'   Raggio di ricerca:            {SEARCH_RADIUS_M} m')
print()
print(f"  {'CLASSE':<30} {'COLONNA EXCEL':<22} {'TOTALE':>8} {'MEDIA':>8}")
print('  ' + '-' * 72)

for col_name, api_val in MAPILLARY_CLASSES.items():
    if col_name not in df_tot.columns:
        continue
    tot = int(df_tot[col_name].sum())
    avg = df_tot[col_name].mean()
    print(f"  {api_val.split('--')[-1]:<30} {col_name:<22} {tot:>8} {avg:>8.2f}")

print('=' * 72)
print(f'\n💾 File Excel: {OUTPUT_EXCEL_PATH}')
print(f'   Checkpoint:  {CHECKPOINT_PATH}')
print()
print('ℹ️  Il checkpoint JSON può essere eliminato solo quando sei sicuro')
print('   che l\'elaborazione è completa e l\'Excel è corretto.')